In [7]:
# ==== remove TensorFlow/Keras parts above and use this instead ====
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from mravens_icons import *

# ---------- helpers ----------
def one_hot(y, n):
    Y = np.zeros((len(y), n), dtype=float)
    Y[np.arange(len(y)), y] = 1.0
    return Y

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def ce_loss(P, Y):
    eps = 1e-12
    return -np.mean(np.sum(Y * np.log(P + eps), axis=1))

def accuracy_from_probs(P, Y):
    return np.mean(np.argmax(P, axis=1) == np.argmax(Y, axis=1))

def grad_W(X, P, Y, W, l2=0.0):
    N = X.shape[0]
    g = (X.T @ (P - Y)) / N
    if l2 > 0.0:
        g += l2 * W
    return g

def train_epoch(W, X, Y, lr=0.1, batch_size=32, l2=0.0, clip=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.permutation(X.shape[0])
    for s in range(0, len(idx), batch_size):
        b = idx[s:s+batch_size]
        Z = X[b] @ W
        P = softmax(Z)
        G = grad_W(X[b], P, Y[b], W, l2=l2)
        if clip is not None:
            gnorm = np.linalg.norm(G)
            if gnorm > clip:
                G *= (clip / (gnorm + 1e-12))
        W -= lr * G
    return W

# ---------- data ----------
data = load_wine()
X, y = data['data'], data['target']
K = int(np.max(y) + 1)

# min-max norm (same spirit as your code)
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-12)
Y = one_hot(y, K)

X_train, X_test, Y_train, Y_test, ytrain, ytest = train_test_split(
    X_norm, Y, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- rate coding (unchanged from your idea) ----------
f_sample = 32
X_rate_train = (X_train * f_sample).astype(int)
X_rate_test  = (X_test  * f_sample).astype(int)

def build_rate_tensor(X_rate):
    N, D = X_rate.shape
    X_rate_data = np.zeros((N, D, f_sample), dtype=int)
    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                step = max(1, round(f_sample / feature))
                X_rate_data[m, n, 0:-1:step] = 1
    return X_rate_data

X_rate_data_train = build_rate_tensor(X_rate_train)
X_rate_data_test  = build_rate_tensor(X_rate_test)

# ---------- your simulator (we assume you have mravens_icons in scope) ----------
# IMPORTANT: we add return_counts=True so we can get spike counts per output neuron
def simulate(X_rate_data, W, f_sample, st_point, n_id, type_id, abs_period, return_counts=False):
    y_pred = []
    counts = [] if return_counts else None
    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], W)   # uses W as synapses
        for nid, tpe, k in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, tpe)
            net.neuron[nid].abs_period = int(k)
        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()
        sc = np.sum(event.spikes_, axis=0)  # spike counts per output neuron
        y_pred.append(int(np.argmax(sc)))
        if return_counts:
            counts.append(sc)
    if return_counts:
        return np.array(y_pred), np.vstack(counts)
    return np.array(y_pred)

# ---------- initialize linear softmax weights (no bias) ----------
D = X_train.shape[1]
rng = np.random.default_rng(0)
# Glorot std
std = np.sqrt(2.0 / (D + K))
W = rng.normal(0.0, std, size=(D, K))

# ---------- pre-train classifier (optional but helps) ----------
for _ in range(10):
    Z = X_train @ W
    P = softmax(Z)
    W = train_epoch(W, X_train, Y_train, lr=0.1, batch_size=32, l2=1e-4, clip=5.0, rng=rng)

# ---------- supervised bio-inspired refractory update + joint learning ----------
abs_period = rng.integers(low=2, high=max(4, f_sample // 8), size=K)  # heterogeneity
T_min, T_max = 1, f_sample
eta_T = 0.05          # refractory step size (task supervised)
mu_T  = 0.001         # homeostasis gain
target_spikes = 0.2 * f_sample  # per-sample target per neuron

outer_iters   = 100
inner_epochs  = 1     # tiny joint step on weights each outer iter
lr_W          = 0.05
l2_W          = 1e-4

acc_train_list, acc_test_list, abs_period_list = [], [], []

for it in range(outer_iters):
    # --- simulate SNN with current W and refractory ---
    y_pred_train, spk_train = simulate(
        X_rate_data_train, W, f_sample,
        st_point=np.arange(X_train.shape[1]), n_id=list(range(K)),
        type_id=["output"] * K, abs_period=abs_period, return_counts=True
    )
    y_pred_test = simulate(
        X_rate_data_test, W, f_sample,
        st_point=np.arange(X_test.shape[1]), n_id=list(range(K)),
        type_id=["output"] * K, abs_period=abs_period, return_counts=False
    )

    acc_train = np.mean(y_pred_train == ytrain)
    acc_test  = np.mean(y_pred_test  == ytest)
    acc_train_list.append(acc_train); acc_test_list.append(acc_test)
    abs_period_list.append(abs_period.copy())

    print(f"[Iter {it:02d}] SNN acc train/test: {acc_train:.3f}/{acc_test:.3f} | T={abs_period}")

    # --- supervised error tallies per class ---
    misses = np.zeros(K, dtype=int)        # true=k, predicted!=k  → T_k should DECREASE (more spikes)
    false_alarms = np.zeros(K, dtype=int)  # predicted=k, true!=k → T_k should INCREASE (fewer spikes)
    for k in range(K):
        misses[k]       = np.sum((ytrain == k) & (y_pred_train != k))
        false_alarms[k] = np.sum((ytrain != k) & (y_pred_train == k))

    # --- homeostatic term (intrinsic plasticity) ---
    mean_spk = spk_train.mean(axis=0)         # avg per neuron over train
    homeo = mean_spk - target_spikes          # +ve → too spiky (increase T), -ve → underactive (decrease T)

    # --- refractory update (teacher-gated + homeostasis) ---
    dT = eta_T * (false_alarms - misses) + mu_T * homeo
    abs_period = np.clip((abs_period + np.rint(dT)).astype(int), T_min, T_max)

    # --- joint learning: quick supervised backprop on weights (no TF) ---
    for _ in range(inner_epochs):
        W = train_epoch(W, X_train, Y_train, lr=lr_W, batch_size=32, l2=l2_W, clip=5.0, rng=rng)

# Final linear model (for reference)
P_train = softmax(X_train @ W)
P_test  = softmax(X_test  @ W)
print(f"Linear softmax (no spikes) acc train/test: "
      f"{accuracy_from_probs(P_train, Y_train):.3f}/{accuracy_from_probs(P_test, Y_test):.3f}")

[Iter 00] SNN acc train/test: 0.697/0.639 | T=[3 3 3]
[Iter 01] SNN acc train/test: 0.782/0.806 | T=[5 1 3]
[Iter 02] SNN acc train/test: 0.838/0.833 | T=[4 2 3]
[Iter 03] SNN acc train/test: 0.831/0.806 | T=[4 2 3]
[Iter 04] SNN acc train/test: 0.838/0.833 | T=[4 2 3]
[Iter 05] SNN acc train/test: 0.831/0.833 | T=[4 2 3]
[Iter 06] SNN acc train/test: 0.852/0.861 | T=[3 2 3]
[Iter 07] SNN acc train/test: 0.845/0.861 | T=[3 2 3]
[Iter 08] SNN acc train/test: 0.845/0.889 | T=[3 2 3]
[Iter 09] SNN acc train/test: 0.852/0.889 | T=[3 2 3]
[Iter 10] SNN acc train/test: 0.859/0.889 | T=[3 2 3]
[Iter 11] SNN acc train/test: 0.859/0.889 | T=[3 2 3]
[Iter 12] SNN acc train/test: 0.873/0.889 | T=[3 2 3]
[Iter 13] SNN acc train/test: 0.873/0.917 | T=[3 2 3]
[Iter 14] SNN acc train/test: 0.880/0.917 | T=[3 2 3]
[Iter 15] SNN acc train/test: 0.880/0.917 | T=[3 2 3]
[Iter 16] SNN acc train/test: 0.894/0.944 | T=[3 2 3]
[Iter 17] SNN acc train/test: 0.859/0.917 | T=[2 2 3]
[Iter 18] SNN acc train/test

In [8]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from mravens_icons import *  # your simulator

# ---------- helpers ----------
def one_hot(y, n):
    Y = np.zeros((len(y), n), dtype=float)
    Y[np.arange(len(y)), y] = 1.0
    return Y

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def build_rate_tensor(X_rate, f_sample):
    N, D = X_rate.shape
    X_rate_data = np.zeros((N, D, f_sample), dtype=int)
    for m, sample in enumerate(X_rate):
        for n, feature in enumerate(sample):
            if feature > 0:
                step = max(1, round(f_sample / feature))
                X_rate_data[m, n, 0:-1:step] = 1
    return X_rate_data

# ---------- data ----------
data = load_wine()
X, y = data['data'], data['target']
K = int(np.max(y) + 1)

# min-max norm
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0) + 1e-12)
Y = one_hot(y, K)

X_tr, X_te, Y_tr, Y_te, ytr, yte = train_test_split(
    X, Y, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- rate coding ----------
f_sample = 32
X_rate_tr = (X_tr * f_sample).astype(int)
X_rate_te = (X_te * f_sample).astype(int)
Xr_tr = build_rate_tensor(X_rate_tr, f_sample)   # [N,D,T]
Xr_te = build_rate_tensor(X_rate_te, f_sample)

N_tr, D, T = Xr_tr.shape

# ---------- surrogate forward/backward through time ----------
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

def surrogate_forward(Xr, W, b, Tref, theta=0.0, beta=1.0, tau=2.0):
    """
    Xr: [N,D,T], W:[D,K], b:[K], Tref:[K]
    Returns: S:[N,K], cache(dict) with per-time buffers for BPTT
    """
    N, D, Tlen = Xr.shape
    K = W.shape[1]
    # I[n,k,t] = sum_d X[n,d,t] * W[d,k] + b[k]
    I = np.tensordot(Xr, W, axes=([1],[0])) + b  # [N,T,K]
    I = np.transpose(I, (0,2,1))                 # [N,K,T]
    a_pre = (I - theta) / beta
    a = sigmoid(a_pre)                           # drive
    u = np.zeros((N, K))                         # time-since-last spike
    p = np.zeros((N, K, Tlen))
    r = np.zeros_like(p)                         # refractory activation σ((T-u)/τ)
    mask = np.zeros_like(p)
    u_hist = np.zeros_like(p)                    # u at time t (before update)
    for t in range(Tlen):
        u_hist[:,:,t] = u
        z = (Tref[None,:,None] - u[:, :, None]) / tau  # broadcast to [N,K,1]
        r[:,:,t] = sigmoid(z[:,:,0])                   # [N,K]
        mask[:,:,t] = 1.0 - r[:,:,t]
        p[:,:,t] = a[:,:,t] * mask[:,:,t]
        u = (1.0 - p[:,:,t]) * (u + 1.0)              # soft reset/increment
    S = np.sum(p, axis=2)                              # [N,K]
    cache = dict(I=I, a=a, p=p, r=r, mask=mask, u_hist=u_hist,
                 beta=beta, tau=tau, Tref=Tref, theta=theta, Xr=Xr)
    return S, cache

def surrogate_backward(Y, S, cache, W, l2w=0.0):
    """
    Computes grads dW, db, dTref via BPTT.
    """
    I = cache['I']         # [N,K,T]
    a = cache['a']         # [N,K,T]
    p = cache['p']         # [N,K,T]
    r = cache['r']
    mask = cache['mask']
    u_hist = cache['u_hist']  # [N,K,T]
    Xr = cache['Xr']       # [N,D,T]
    beta = cache['beta']; tau = cache['tau']; Tref = cache['Tref']
    N, K, Tlen = p.shape
    D = Xr.shape[1]

    # dL/dS from CE on softmax(S)
    P = softmax(S)
    dS = (P - Y) / N    # [N,K]

    # Accumulators
    dI = np.zeros_like(I)      # [N,K,T]
    dT = np.zeros((K,), dtype=float)
    db = np.zeros((K,), dtype=float)
    dW = np.zeros((D, K), dtype=float)

    # BPTT vars
    du_next = np.zeros((N, K))   # dL/du_{t+1}
    dp_rec = np.zeros((N, K))    # extra dL/dp_t coming from u_{t+1} dependency

    for t in reversed(range(Tlen)):
        # total dL/dp_t = dS + (from recurrence)
        dp = dS + dp_rec                         # [N,K]

        # p = a * mask
        da = dp * mask[:,:,t]                   # dL/da_t
        dmask = dp * a[:,:,t]                   # dL/dmask_t

        # mask = 1 - r
        dr = -dmask

        # r = σ(z), z=(T - u)/tau
        r_t = r[:,:,t]
        sigp = r_t * (1.0 - r_t)                # σ'(z)
        # contributions to T and u
        dT += np.sum(dr * sigp * (1.0 / tau), axis=0)
        du = dr * sigp * (-1.0 / tau)           # dL/du_t (from r)

        # a = σ((I - θ)/β)
        a_t = a[:,:,t]
        da_pre = da * (a_t * (1.0 - a_t)) * (1.0 / beta)   # dL/dI_t
        dI[:,:,t] += da_pre

        # u_{t+1} = (1 - p_t) * (u_t + 1)
        # backprop to u_t and p_t from next
        du_from_next = du_next * (1.0 - p[:,:,t])          # ∂u_{t+1}/∂u_t
        dp_from_next = du_next * (-(u_hist[:,:,t] + 1.0))  # ∂u_{t+1}/∂p_t
        du_total = du + du_from_next
        dp_rec = dp_from_next
        du_next = du_total

    # dW, db: sum over N,T   (I[n,k,t] = sum_d Xr[n,d,t]*W[d,k] + b[k])
    # dI has shape [N,K,T]; gradient wrt W is tensordot over n,t
    dW = np.tensordot(Xr, dI, axes=([0,2],[0,2]))   # [D,K]
    db = np.sum(dI, axis=(0,2))                     # [K]

    if l2w > 0.0:
        dW += l2w * W

    return dW, db, dT

# ---------- training loop (surrogate SGD) ----------
rng = np.random.default_rng(0)
W = rng.normal(0.0, np.sqrt(2.0/(D+K)), size=(D,K))
b = np.zeros((K,), dtype=float)
# start Tref modest; keep > 0
Tref = rng.integers(low=2, high=max(4, f_sample//8), size=K).astype(float)

epochs = 60
batch = 32
lr_w, lr_b, lr_t = 0.05, 0.05, 0.03
l2w = 1e-4
beta, tau, theta = 1.0, 2.0, 0.0
Tmin, Tmax = 1.0, float(f_sample)

idx_all = np.arange(N_tr)

for ep in range(1, epochs+1):
    rng.shuffle(idx_all)
    for s in range(0, N_tr, batch):
        ids = idx_all[s:s+batch]
        S, cache = surrogate_forward(
            Xr_tr[ids], W, b, Tref, theta=theta, beta=beta, tau=tau
        )
        dW, db_g, dT = surrogate_backward(Y_tr[ids], S, cache, W, l2w=l2w)
        # SGD step
        W -= lr_w * dW
        b -= lr_b * db_g
        Tref -= lr_t * dT
        # clamp T
        Tref = np.clip(Tref, Tmin, Tmax)

    # monitor
    S_tr, _ = surrogate_forward(Xr_tr, W, b, Tref, theta, beta, tau)
    S_te, _ = surrogate_forward(Xr_te, W, b, Tref, theta, beta, tau)
    acc_tr = np.mean(np.argmax(S_tr, axis=1) == ytr)
    acc_te = np.mean(np.argmax(S_te, axis=1) == yte)
    if ep % 10 == 0 or ep == 1:
        print(f"[ep {ep:02d}] surrogate acc train/test: {acc_tr:.3f}/{acc_te:.3f}  | T={np.round(Tref,2)}")

# ---------- evaluate on your real simulator ----------
abs_period = np.clip(np.rint(Tref).astype(int), 1, f_sample)
def simulate(X_rate_data, W, f_sample, st_point, n_id, type_id, abs_period, return_counts=False):
    y_pred = []
    counts = [] if return_counts else None
    for i in range(len(X_rate_data)):
        net = create_custom_network()
        net.add_stimuli(st_point, n_id, X_rate_data[i], W)
        for nid, tpe, k in zip(n_id, type_id, abs_period):
            net.add_neuron(nid, tpe)
            net.neuron[nid].abs_period = int(k)
        proc = MRAVENS(-5, 5)
        event = process_event(f_sample, net, proc)
        event.apply_spike()
        sc = np.sum(event.spikes_, axis=0)
        y_pred.append(int(np.argmax(sc)))
        if return_counts: counts.append(sc)
    if return_counts:
        return np.array(y_pred), np.vstack(counts)
    return np.array(y_pred)

y_pred_tr = simulate(Xr_tr, W, f_sample, st_point=np.arange(D),
                     n_id=list(range(K)), type_id=["output"]*K, abs_period=abs_period)
y_pred_te = simulate(Xr_te, W, f_sample, st_point=np.arange(D),
                     n_id=list(range(K)), type_id=["output"]*K, abs_period=abs_period)

print(f"REAL sim acc train/test: {np.mean(y_pred_tr==ytr):.3f}/{np.mean(y_pred_te==yte):.3f} | abs_period={abs_period}")

[ep 01] surrogate acc train/test: 0.507/0.417  | T=[2.99 2.02 3.  ]
[ep 10] surrogate acc train/test: 0.923/0.861  | T=[3.02 1.96 2.95]
[ep 20] surrogate acc train/test: 0.972/0.917  | T=[3.   1.89 2.9 ]
[ep 30] surrogate acc train/test: 0.958/0.917  | T=[2.98 1.84 2.86]
[ep 40] surrogate acc train/test: 0.958/0.944  | T=[2.95 1.79 2.82]
[ep 50] surrogate acc train/test: 0.965/0.917  | T=[2.91 1.74 2.79]
[ep 60] surrogate acc train/test: 0.958/0.944  | T=[2.88 1.7  2.76]
REAL sim acc train/test: 0.690/0.694 | abs_period=[3 2 3]
